# University Course Assistant using Retrieval-Augmented Generation (RAG)

## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) system using
custom university course data.

The system allows users to ask questions about university courses. It uses
Sentence Transformers to convert the data and user questions into numerical
embeddings, Qdrant as a vector database to retrieve relevant information,
and Llama 3.2 running through Ollama to generate the final answer.

## Problem Statement

Large Language Models may not know specific information about a university's
courses. Therefore, this project uses a custom dataset containing university
course information and retrieves relevant information before asking the LLM
to generate an answer.

## Technologies Used

- Python
- Sentence Transformers
- Qdrant
- Ollama
- Llama 3.2 1B
- Google Colab

## RAG Pipeline

User Question
→ Sentence Transformer
→ Vector Embedding
→ Qdrant Similarity Search
→ Relevant Context
→ Llama 3.2
→ Final Answer

## What is Retrieval-Augmented Generation?

Retrieval-Augmented Generation (RAG) is a technique that combines
information retrieval with text generation.

Instead of asking an LLM to answer a question only using its
pre-trained knowledge, RAG first retrieves relevant information
from an external data source.

The retrieved information is then provided to the LLM as context,
which helps the LLM generate an answer based on the provided data.

In this project:

1. The user's question is converted into an embedding.
2. Qdrant searches for documents with similar embeddings.
3. The most relevant documents are retrieved.
4. The retrieved documents are added to a prompt.
5. Llama 3.2 uses the retrieved context to generate the answer.

In [2]:
!pip install qdrant-client sentence-transformers openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 13.1 MB/s eta 0:00:00


DATASET

In [3]:
documents = [
    {
        "title": "Machine Learning",
        "content": "Machine Learning introduces supervised learning, unsupervised learning, regression, classification, clustering and basic machine learning algorithms."
    },
    {
        "title": "Database Systems",
        "content": "Database Systems covers relational databases, SQL, database design, normalization, transactions and database management."
    },
    {
        "title": "Principles of Programming Languages",
        "content": "Principles of Programming Languages introduces programming language concepts, syntax, semantics, types, functional programming and programming paradigms."
    },
    {
        "title": "Organizational Behaviour",
        "content": "Organizational Behaviour studies individual and group behaviour in organizations, motivation, leadership, communication and organizational culture."
    },
    {
        "title": "Computer Networks",
        "content": "Computer Networks covers networking concepts, communication protocols, IP addresses, routing, network security and data transmission."
    }
]

In [4]:
print(documents)

[{'title': 'Machine Learning', 'content': 'Machine Learning introduces supervised learning, unsupervised learning, regression, classification, clustering and basic machine learning algorithms.'}, {'title': 'Database Systems', 'content': 'Database Systems covers relational databases, SQL, database design, normalization, transactions and database management.'}, {'title': 'Principles of Programming Languages', 'content': 'Principles of Programming Languages introduces programming language concepts, syntax, semantics, types, functional programming and programming paradigms.'}, {'title': 'Organizational Behaviour', 'content': 'Organizational Behaviour studies individual and group behaviour in organizations, motivation, leadership, communication and organizational culture.'}, {'title': 'Computer Networks', 'content': 'Computer Networks covers networking concepts, communication protocols, IP addresses, routing, network security and data transmission.'}]


In [5]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
texts = [document["content"] for document in documents]
print(texts)

['Machine Learning introduces supervised learning, unsupervised learning, regression, classification, clustering and basic machine learning algorithms.', 'Database Systems covers relational databases, SQL, database design, normalization, transactions and database management.', 'Principles of Programming Languages introduces programming language concepts, syntax, semantics, types, functional programming and programming paradigms.', 'Organizational Behaviour studies individual and group behaviour in organizations, motivation, leadership, communication and organizational culture.', 'Computer Networks covers networking concepts, communication protocols, IP addresses, routing, network security and data transmission.']


In [9]:
embeddings = encoder.encode(texts)
print(embeddings[0])

[-2.26619914e-02 -8.71723220e-02  5.54852448e-02  1.28613843e-03
  5.90975769e-02 -6.37285365e-03 -9.28902552e-02 -7.41883740e-02
 -9.73249376e-02 -6.43092534e-03 -2.85555888e-02 -3.02673765e-02
 -1.52422683e-02 -3.21382545e-02 -4.49846201e-02 -2.99739428e-02
  5.63696725e-04 -4.06284072e-02 -5.70698306e-02 -1.12992480e-01
  5.51620983e-02  9.25885234e-03 -6.95421919e-02  3.86029482e-02
  3.00187133e-02  1.89171806e-02  7.71559328e-02  3.13144103e-02
  1.49190985e-02  1.38114020e-02 -9.56104416e-03 -3.20016667e-02
  5.53707331e-02  5.03421053e-02 -8.71124268e-02  5.79594588e-03
  1.91285871e-02  5.66267036e-02 -2.92606577e-02  2.58185323e-02
 -1.93562005e-02 -8.19515362e-02  6.88400399e-03  3.01574823e-04
  1.34407222e-01  8.86537284e-02 -9.46409628e-02 -6.61727786e-02
 -2.33887415e-02 -5.15131699e-03 -7.14663565e-02 -2.16865577e-02
 -1.33486725e-02  7.25644175e-03 -3.94387683e-03  1.37746688e-02
  4.80403304e-02  2.89730374e-02  5.63928932e-02 -3.08547243e-02
  1.89217012e-02 -1.03419

In [10]:
print(embeddings.shape)

(5, 384)


In [11]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(":memory:")

In [12]:
from qdrant_client.models import VectorParams, Distance

qdrant.create_collection(
    collection_name="university_courses",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

True

In [13]:
from qdrant_client.models import PointStruct

points = []

for i, (document, embedding) in enumerate(zip(documents, embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload=document
        )
    )

In [14]:
qdrant.upsert(
    collection_name="university_courses",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [17]:
question = "What topics are covered in machine learning?"


In [18]:
query_vector = encoder.encode(question).tolist()

In [26]:
hits = qdrant.query_points(
    collection_name="university_courses",
    query=query_vector,
    limit=3
)

for hit in hits.points:
    print(hit.payload)
    print("Score:", hit.score)
    print()


{'title': 'Machine Learning', 'content': 'Machine Learning introduces supervised learning, unsupervised learning, regression, classification, clustering and basic machine learning algorithms.'}
Score: 0.6791456731975115

{'title': 'Computer Networks', 'content': 'Computer Networks covers networking concepts, communication protocols, IP addresses, routing, network security and data transmission.'}
Score: 0.35276805482286333

{'title': 'Database Systems', 'content': 'Database Systems covers relational databases, SQL, database design, normalization, transactions and database management.'}
Score: 0.2811765942209682



In [32]:
context = "\n".join(
    hit.payload["content"]
    for hit in hits.points
)
print(context)

Machine Learning introduces supervised learning, unsupervised learning, regression, classification, clustering and basic machine learning algorithms.
Computer Networks covers networking concepts, communication protocols, IP addresses, routing, network security and data transmission.
Database Systems covers relational databases, SQL, database design, normalization, transactions and database management.


In [33]:
prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer using only the information provided in the context.
"""

In [58]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 19 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,094 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [59]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [60]:
!ollama --version

In [61]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Ollama server started.")

Ollama server started.


In [62]:
import requests

response = requests.get("http://127.0.0.1:11434")

print("Status code:", response.status_code)
print(response.text)

Status code: 200
Ollama is running


In [63]:
!ollama pull llama3.2:1b

In [64]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:1b    baf6a787fdff    1.3 GB    8 seconds ago    


In [65]:
!ollama run llama3.2:1b "What is machine learning? Explain in one sentence."

Machine learning is a subset of artificial intelligence that involves train
training algorithms to learn from data, make predictions or decisions, and 
improve their performance over time without being explicitly programmed.



In [66]:
import requests

In [67]:
response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2:1b",
        "prompt": prompt,
        "stream": False
}
)

In [68]:
result = response.json()

print(result["response"])

Machine Learning covers supervised learning, unsupervised learning, regression, classification, clustering, and basic machine learning algorithms.


TESTING

In [69]:
question = "What topics are covered in Machine Learning?"

In [70]:
query_vector = encoder.encode(question).tolist()

hits = qdrant.query_points(
    collection_name="university_courses",
    query=query_vector,
    limit=3
)

In [71]:
context = "\n".join(
    hit.payload["content"]
    for hit in hits.points
)

print(context)

Machine Learning introduces supervised learning, unsupervised learning, regression, classification, clustering and basic machine learning algorithms.
Computer Networks covers networking concepts, communication protocols, IP addresses, routing, network security and data transmission.
Database Systems covers relational databases, SQL, database design, normalization, transactions and database management.


In [72]:
prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer using only the information provided in the context.
"""

In [73]:
response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2:1b",
        "prompt": prompt,
        "stream": False
    }
)

result = response.json()

print(result["response"])

Based on the context provided, Machine Learning is covered by the topics of:

1. Supervised learning
2. Unsupervised learning
3. Regression
4. Classification
5. Clustering
6. Basic machine learning algorithms


In [74]:
def ask_rag(question):
    # Step 1: Convert the user's question into a vector
    query_vector = encoder.encode(question).tolist()

    # Step 2: Search Qdrant for the most relevant information
    hits = qdrant.query_points(
        collection_name="university_courses",
        query=query_vector,
        limit=3
    )

    # Step 3: Extract the text from the retrieved documents
    context = "\n".join(
        hit.payload["content"]
        for hit in hits.points
    )

    # Step 4: Create a prompt containing the retrieved information
    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer using only the information provided in the context.
"""

    # Step 5: Send the prompt to the local LLM
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": prompt,
            "stream": False
        }
    )

    # Step 6: Extract and return the LLM's answer
    result = response.json()

    return result["response"]

## Testing the RAG System

The system was tested using several questions related to the
custom university course dataset.

In [75]:
answer = ask_rag(
    "What topics are covered in Machine Learning?"
)

print(answer)

Based on the provided context, the following topics are covered in Machine Learning:

1. Supervised learning
2. Unsupervised learning
3. Regression
4. Classification
5. Clustering


In [76]:
answer = ask_rag(
    "Which course covers SQL and normalization?"
)

print(answer)

Based on the provided context, the course that covers SQL and normalization is:

Database Systems covers relational databases, SQL, database design, normalization, transactions, and database management.


In [77]:
answer = ask_rag(
    "What topics are covered in Organizational Behaviour?"
)

print(answer)

Based on the provided context, the topics covered in Organizational Behaviour are:

1. Individual behaviour
2. Group behaviour
3. Motivation
4. Leadership
5. Communication
6. Organizational culture


In [78]:
answer = ask_rag(
    "What does the Computer Networks course cover?"
)

print(answer)

The Computer Networks course covers networking concepts, communication protocols, IP addresses, routing, network security, and data transmission.
